# Run-centric graph-state transition visualization

This notebook analyzes **one hallucinated response at a time and one hallucination span at a time**. It fixes the main confounds in the earlier rich t-SNE workflow: generator provenance is explicit, position effects are residualized against same-generator correct controls, correlated hand-crafted features are block-balanced, cumulative history-lag bins are made disjoint, and each error onset is compared with a matched correct-control transition null.


In [ ]:
from pathlib import Path
import sys

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == 'notebooks':
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

from transition_visualization import TransitionSampleVisualizer

DATA_ROOT = Path('/share/home/tm902089733300000/a903202310/lys/data/RAGTruth/model_traces/llama31_8b/test')
OUTPUT_ROOT = REPO_ROOT / 'outputs' / 'single_sample_transition'

# RAGTruth does not contain Llama-3.1-8B-generated responses. Leave None to use
# the generator recorded in the canonical manifest, or set e.g. 'llama-2-7b-chat'.
GENERATOR_MODEL = None
ERROR_SAMPLE_ID = '10071'
ERROR_INDEX = 0
PRE_WINDOW = 10
POST_WINDOW = 10
LOCAL_RADIUS = 18
MAX_CONTROLS = 32


In [ ]:
viewer = TransitionSampleVisualizer(
    DATA_ROOT,
    output_root=OUTPUT_ROOT,
    generator_model=GENERATOR_MODEL,
    random_state=0,
)
provenance = viewer.provenance()
provenance


## Verify model provenance

`observer_model` is the model whose internal attention is being analyzed. `selected_generator_model` is the model that originally wrote the RAGTruth response text. If they differ, this is a teacher-forced/proxy analysis rather than a same-generator internal-state experiment.


In [ ]:
sample_id = (
    str(ERROR_SAMPLE_ID)
    if ERROR_SAMPLE_ID is not None
    else viewer.error_sample_ids[ERROR_INDEX]
)
sample = viewer.dataset[sample_id]
print('sample_id:', sample_id)
print('observer_model:', sample.observer_model)
print('generator_model:', sample.generator_model)
print('task_type:', sample.task_type, 'data_source:', sample.data_source)
print('positive_runs:', viewer.labels.positive_runs(sample_id))
viewer.runs(sample_id, pre_window=PRE_WINDOW, post_window=POST_WINDOW)


## Run-centric analysis

Each hallucination span is handled independently. For nearby spans such as sample 10071, the second run may have only a very short clean pre-error segment; the metadata records this instead of silently mixing the previous hallucination into `pre_error`.


In [ ]:
result = viewer.visualize(
    sample_id,
    pre_window=PRE_WINDOW,
    post_window=POST_WINDOW,
    local_radius=LOCAL_RADIUS,
    max_controls=MAX_CONTROLS,
)
result['metadata']


## Inspect one run numerically

The empirical percentiles compare the observed pre→error state shift and onset transition score against pseudo-onsets at matched normalized positions in fully correct responses from the **same generator**.


In [ ]:
RUN_INDEX = 0
viewer.run_metrics(
    sample_id,
    run_index=RUN_INDEX,
    pre_window=PRE_WINDOW,
    post_window=POST_WINDOW,
    max_controls=MAX_CONTROLS,
)
